In [ ]:
!pip install torch torchvision pillow tqdm optuna numpy

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import optuna

In [ ]:
# -------------------------
# Utils
# -------------------------

def psnr(x, y):
    mse = torch.mean((x - y) ** 2)
    if mse == 0:
        return torch.tensor(100.0, device=x.device)
    return 20 * torch.log10(1.0 / torch.sqrt(mse))


def adjust_lr(optimizer, epoch):
    if epoch < 25:
        lr = 1e-3
    elif epoch < 55:
        lr = 5e-4
    else:
        lr = 1e-4
    for p in optimizer.param_groups:
        p['lr'] = lr

# -------------------------
# Dataset
# -------------------------

class StegDataset(Dataset):
    def __init__(self, cover_dir, secret_dir):
        self.cover_imgs = sorted([os.path.join(cover_dir, f) for f in os.listdir(cover_dir)])
        self.secret_imgs = sorted([os.path.join(secret_dir, f) for f in os.listdir(secret_dir)])
        self.t = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.cover_imgs)

    def __getitem__(self, idx):
        cover = self.t(Image.open(self.cover_imgs[idx]).convert('RGB'))
        secret = self.t(Image.open(self.secret_imgs[idx]).convert('RGB'))
        return cover, secret


In [ ]:
# -------------------------
# Model Blocks
# -------------------------


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k=3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, k, padding=k // 2),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )  
    def forward(self, x):
        return self.block(x)

# -------------------------
# Encoder
# -------------------------

class Encoder(nn.Module):
    def __init__(self, depth=6, base_c=64, k=3, alpha=0.1):
        super().__init__()
        self.alpha = alpha
        layers = [ConvBlock(6, base_c, k)]
        for _ in range(depth - 2):
            layers.append(ConvBlock(base_c, base_c, k))
        layers.append(nn.Conv2d(base_c, 3, k, padding=k // 2))
        self.net = nn.Sequential(*layers)

    def forward(self, cover, secret):
        x = torch.cat([cover, secret], dim=1)
        residual = self.net(x)
        stego = torch.clamp(cover + self.alpha * residual, 0, 1)
        return stego

# -------------------------
# Decoder
# -------------------------

class Decoder(nn.Module):
    def __init__(self, depth=6, base_c=64, k=3):
        super().__init__()
        layers = [ConvBlock(3, base_c, k)]
        for _ in range(depth - 2):
            layers.append(ConvBlock(base_c, base_c, k))
        layers.append(nn.Conv2d(base_c, 3, k, padding=k // 2))
        self.net = nn.Sequential(*layers)

    def forward(self, stego):
        return torch.clamp(self.net(stego), 0, 1)

# -------------------------
# Full Model
# -------------------------

class HyperStegNet(nn.Module):
    def __init__(self, depth=6, base_c=64, k=3, alpha=0.1):
        super().__init__()
        self.encoder = Encoder(depth, base_c, k, alpha)
        self.decoder = Decoder(depth, base_c, k)

    def forward(self, cover, secret):
        stego = self.encoder(cover, secret)
        recon = self.decoder(stego)
        return stego, recon


In [ ]:
# -------------------------
# Training + Validation
# -------------------------

def train(model, train_loader, val_loader, epochs, lambda_c, lambda_s, trial=None, save_path=None):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    best_score = 0.0
    patience, patience_ctr = 10, 0
    resume_path="/kaggle/input/f1/pytorch/default/1/final.pth"
    start_epoch = 0

    # ========= RESUME FROM CHECKPOINT =========
    if resume_path is not None and os.path.exists(resume_path):
        checkpoint = torch.load(resume_path, map_location=device)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        start_epoch = checkpoint['epoch']
        best_score = checkpoint.get('best_score', 0.0)
        patience_ctr = checkpoint.get('patience_ctr', 0)

        print(f"✅ Resumed from epoch {start_epoch}, best score = {best_score:.2f}")

    for epoch in range(start_epoch, epochs):
        adjust_lr(optimizer, epoch)
        model.train()

        for cover, secret in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False):
            cover, secret = cover.to(device), secret.to(device)
            stego, recon = model(cover, secret)

            loss = lambda_c * loss_fn(stego, cover) + lambda_s * loss_fn(recon, secret)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # -------- Validation --------
        model.eval()
        psnr_c, psnr_s, n = 0.0, 0.0, 0

        with torch.no_grad():
            for cover, secret in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False):
                cover, secret = cover.to(device), secret.to(device)
                stego, recon = model(cover, secret)
                psnr_c += psnr(stego, cover).item()
                psnr_s += psnr(recon, secret).item()
                n += 1

        avg_psnr_c = psnr_c / n
        avg_psnr_s = psnr_s / n
        score = avg_psnr_c + avg_psnr_s

        # ---- Optuna reporting ----
        if trial is not None:
            trial.report(score, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        # ---- Early stopping + save FULL CHECKPOINT ----
        if score > best_score:
            best_score = score
            patience_ctr = 0
            if save_path:
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'depth': model.encoder.net.__len__() + 1,
                    'base_c': model.encoder.net[0].block[0].out_channels,
                    'kernel_size': model.encoder.net[0].block[0].kernel_size[0],
                    'alpha': model.encoder.alpha,
                    'lambda_c': lambda_c,
                    'lambda_s': lambda_s,
                    'best_score': best_score
                }, save_path)
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            break

    return avg_psnr_c, avg_psnr_s

In [ ]:
# -------------------------
# Optuna Objective
# -------------------------


def objective(trial):
    best_psnr=65.0529756011963
    depth = trial.suggest_int('depth', 4, 10)
    base_c = trial.suggest_categorical('base_c', [32, 64, 96, 128])
    k = trial.suggest_categorical('kernel', [3, 5])
    alpha = trial.suggest_float('alpha', 0.01, 0.2)
    lambda_c = trial.suggest_float('lambda_c', 0.5, 5.0)
    lambda_s = trial.suggest_float('lambda_s', 1.0, 10.0)
    
    
    # model = HyperStegNet(depth, base_c, k, alpha)
    
    
    # psnr_c, psnr_s = train(
    #     model,
    #     train_loader,
    #     val_loader,
    #     epochs=20,
    #     lambda_c=lambda_c,
    #     lambda_s=lambda_s,
    #     trial=trial,
    #     save_path=f"/kaggle/working/trial_{trial.number}.pth"
    # )

    model = HyperStegNet(7, 96,  3, 0.04104349615062547)    
    
    psnr_c, psnr_s = train(
        model,
        train_loader,
        val_loader,
        epochs=80,
        lambda_c=3.1903168120272407,
        lambda_s=2.554258904477149,
        trial=trial,
        save_path="/kaggle/working/final.pth"
    )
    
    if best_psnr<(psnr_c + psnr_s):
        best_psnr=(psnr_c + psnr_s)
        torch.save({
            'model_state_dict': model.state_dict(),
            'depth': model.encoder.net.__len__() + 1,
            'base_c': model.encoder.net[0].block[0].out_channels,
            'kernel_size': model.encoder.net[0].block[0].kernel_size[0],
            'alpha': model.encoder.alpha,
            'lambda_c': lambda_c,
            'lambda_s': lambda_s,
            'best_score': best_psnr
        }, "/kaggle/working/best_model.pth")


    return psnr_c + psnr_s

In [ ]:
# -------------------------
# Run Study
# -------------------------

if __name__ == '__main__':
    train_set = StegDataset('/kaggle/input/dncnn1/DnCNN/train/cover', '/kaggle/input/dncnn1/DnCNN/train/secret')
    val_set = StegDataset('/kaggle/input/dncnn1/DnCNN/val/cover', '/kaggle/input/dncnn1/DnCNN/val/secret')

    train_loader = DataLoader(train_set, batch_size=8, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_set, batch_size=8, shuffle=False)

    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
    )

    study.optimize(objective, n_trials=1)

    print('Best Trial:', study.best_trial.number)
    print('Best Params:', study.best_params)
    print('Best Score:', study.best_value)

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from skimage.metrics import structural_similarity as ssim_fn


In [ ]:
def mse(x, y):
    return torch.mean((x - y) ** 2).item()


def psnr(x, y):
    m = mse(x, y)
    if m == 0:
        return 100
    return 20 * np.log10(1.0 / np.sqrt(m))


def ssim(x, y):
    """
    x, y: [B, C, H, W] in range [0,1]
    returns: mean SSIM over batch
    """
    B = x.size(0)
    ssim_vals = []

    for i in range(B):
        xi = x[i].permute(1, 2, 0).detach().cpu().numpy()
        yi = y[i].permute(1, 2, 0).detach().cpu().numpy()

        ssim_vals.append(
            ssim_fn(xi, yi, channel_axis=2, data_range=1.0)
        )

    return sum(ssim_vals) / len(ssim_vals)

def lmse(x, y, window=8):
    # Local MSE (average over sliding windows)
    _, _, H, W = x.shape
    lmse_val = 0
    count = 0

    for i in range(0, H - window + 1, window):
        for j in range(0, W - window + 1, window):
            x_patch = x[:, :, i:i+window, j:j+window]
            y_patch = y[:, :, i:i+window, j:j+window]
            lmse_val += torch.mean((x_patch - y_patch) ** 2)
            count += 1

    return (lmse_val / count).item()


def bpp(cover, stego):
    # Approximate embedding rate (residual energy → bits)
    residual = stego - cover
    mse_val = torch.mean(residual ** 2)
    return float(torch.log2(1 + mse_val * 255**2))


In [ ]:
def evaluate_test(test_loader):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    ckpt = torch.load("trial_0.pth", map_location=device)

    model = HyperStegNet(
        base_c=ckpt['base_c'],
        depth=ckpt['depth']-1,
        k=ckpt['kernel_size'],
        alpha=ckpt['alpha']
    )

    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    model.eval()

    metrics = {
        'stego': {'mse': [], 'psnr': [], 'ssim': [], 'lmse': [], 'bpp': []},
        'secret': {'mse': [], 'psnr': [], 'ssim': [], 'lmse': []}
    }

    with torch.no_grad():
        for cover, secret in tqdm(test_loader, desc="Testing", leave=False):
            cover = cover.to(device)
            secret = secret.to(device)

            stego, recon = model(cover, secret)

            # ----- Cover → Stego -----
            metrics['stego']['mse'].append(mse(cover, stego))
            metrics['stego']['psnr'].append(psnr(cover, stego))
            metrics['stego']['ssim'].append(ssim(cover, stego))
            metrics['stego']['lmse'].append(lmse(cover, stego))
            metrics['stego']['bpp'].append(bpp(cover, stego))

            # ----- Secret → Reconstructed -----
            metrics['secret']['mse'].append(mse(secret, recon))
            metrics['secret']['psnr'].append(psnr(secret, recon))
            metrics['secret']['ssim'].append(ssim(secret, recon))
            metrics['secret']['lmse'].append(lmse(secret, recon))

    # ---- Average results ----
    print("\n===== TEST RESULTS =====")

    print("\nCover → Stego")
    for k, v in metrics['stego'].items():
        print(f"{k.upper():5s}: {np.mean(v):.4f}")

    print("\nSecret → Reconstructed")
    for k, v in metrics['secret'].items():
        print(f"{k.upper():5s}: {np.mean(v):.4f}")


In [ ]:
test_set = StegDataset('HyperCNN/test/cover', 'HyperCNN/test/secret')
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

evaluate_test(test_loader)
